In [4]:
# =============================================================================
# Persona Builder (8 personas) - PATH FIXED VERSION
# =============================================================================
import numpy as np
import pandas as pd
import re
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------------------------------------------------------
# [0] Base path (중요)
# -----------------------------------------------------------------------------
BASE_DIR = Path("../data_csv")  # ← 현재 노트북 위치 기준
# 만약 노트북이 third_week 바로 아래면:
# BASE_DIR = Path("data_csv")

ING_META_PATH   = BASE_DIR / "ingredient_meta.csv"
ING_EMB_PATH    = BASE_DIR / "ingredient_embeddings.npy"
BRAND_TONE_PKL  = BASE_DIR / "brand_analysis_result.pkl"

OUT_NPY  = BASE_DIR / "persona_vectors.npy"
OUT_CSV  = BASE_DIR / "persona_vectors.csv"
OUT_META = BASE_DIR / "persona_meta.csv"

# -----------------------------------------------------------------------------
# Utils
# -----------------------------------------------------------------------------
def normalize(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    return v if n == 0 else (v / n)

def norm_ing(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[\[\]\(\)\{\}]", " ", s)
    s = re.sub(r"[^0-9a-z가-힣\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# -----------------------------------------------------------------------------
# [1] Load ingredient embeddings
# -----------------------------------------------------------------------------
ingredient_meta = pd.read_csv(ING_META_PATH)
ingredient_embeddings = np.load(ING_EMB_PATH).astype(np.float32)

if "ingredient_name" not in ingredient_meta.columns:
    raise ValueError("ingredient_meta.csv에 'ingredient_name' 컬럼이 없음")

ingredient_meta["ingredient_name_norm"] = ingredient_meta["ingredient_name"].map(norm_ing)
ING_DIM = ingredient_embeddings.shape[1]

ingredient_embedding_dict = {}
for i, row in ingredient_meta.iterrows():
    k = row["ingredient_name_norm"]
    if k and k not in ingredient_embedding_dict:
        ingredient_embedding_dict[k] = ingredient_embeddings[i]

ALL_ING_MEAN = np.mean(
    np.stack(list(ingredient_embedding_dict.values())),
    axis=0
).astype(np.float32)

# -----------------------------------------------------------------------------
# Ingredient vector builder (빈 집합 방지)
# -----------------------------------------------------------------------------
def build_ing_vector(keywords):
    vecs = []
    for kw in keywords:
        kw = norm_ing(kw)
        for k, v in ingredient_embedding_dict.items():
            if kw in k:
                vecs.append(v)
    return np.mean(vecs, axis=0).astype(np.float32) if vecs else ALL_ING_MEAN.copy()

# -----------------------------------------------------------------------------
# [2] Load brand tone embeddings
# -----------------------------------------------------------------------------
brand_df = pd.read_pickle(BRAND_TONE_PKL)

tone_col = next(
    c for c in brand_df.columns
    if isinstance(brand_df[c].iloc[0], (list, np.ndarray))
)

BRAND_COL = "브랜드" if "브랜드" in brand_df.columns else "brand"

brand_tone = {
    row[BRAND_COL]: np.array(row[tone_col], dtype=np.float32)
    for _, row in brand_df.iterrows()
}

ALL_TONE_MEAN = np.mean(
    np.stack(list(brand_tone.values())),
    axis=0
).astype(np.float32)

def mean_brand(brands):
    vecs = [brand_tone[b] for b in brands if b in brand_tone]
    return np.mean(vecs, axis=0).astype(np.float32) if vecs else ALL_TONE_MEAN.copy()

# -----------------------------------------------------------------------------
# [3] Persona definitions (8)
# -----------------------------------------------------------------------------
persona_ingredient = {
    "persona_1": ["히알루론산","레티놀","펩타이드","세라마이드"],
    "persona_2": ["병풀","판테놀","알란토인"],
    "persona_3": ["인삼","사포닌"],
    "persona_4": ["비타민c","컬러"],
    "persona_5": ["글리세린","스쿠알란"],
    "persona_6": ["나이아신아마이드"],
    "persona_7": ["히알루론산","병풀","나이아신아마이드"],
    "persona_8": ["병풀","히알루론산"]
}

persona_brand = {
    "persona_1": ["라네즈","이니스프리"],
    "persona_2": ["라네즈"],
    "persona_3": ["설화수","헤라"],
    "persona_4": ["에뛰드"],
    "persona_5": ["이니스프리"],
    "persona_6": ["이니스프리","에뛰드"],
    "persona_7": ["라네즈","이니스프리","에뛰드"],
    "persona_8": ["라네즈","이니스프리"]
}

persona_risk_price = {
    "persona_1": [0.9,0.8,0.7,0.3],
    "persona_2": [0.4,0.3,0.3,0.8],
    "persona_3": [0.6,0.5,0.2,0.2],
    "persona_4": [0.3,0.4,0.5,0.6],
    "persona_5": [0.2,0.2,0.2,0.2],
    "persona_6": [0.1,0.1,0.1,0.9],
    "persona_7": [0.5,0.5,0.5,0.5],
    "persona_8": [0.2,0.2,0.3,0.7]
}

# -----------------------------------------------------------------------------
# [4] Build & Save
# -----------------------------------------------------------------------------
persona_ids = [f"persona_{i}" for i in range(1,9)]
final_vecs, rows = [], []

for pid in persona_ids:
    final = np.concatenate([
        mean_brand(persona_brand[pid]),
        build_ing_vector(persona_ingredient[pid]),
        np.array(persona_risk_price[pid], dtype=np.float32)
    ])
    final_vecs.append(normalize(final))
    rows.append({"persona_id": pid, "final_dim": len(final)})

persona_matrix = np.stack(final_vecs)

np.save(OUT_NPY, persona_matrix)
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
pd.DataFrame({"persona_id": persona_ids}).to_csv(OUT_META, index=False)

print("✅ persona vectors saved:", persona_matrix.shape)
print("cosine similarity:\n", cosine_similarity(persona_matrix))

✅ persona vectors saved: (8, 1540)
cosine similarity:
 [[1.         0.9648693  0.884362   0.8356449  0.9551031  0.8825268
  0.96892595 0.9834153 ]
 [0.9648693  0.9999999  0.8767067  0.80062413 0.89818645 0.8449376
  0.9529542  0.9756224 ]
 [0.884362   0.8767067  1.0000001  0.8548574  0.9052491  0.8240578
  0.89955705 0.8954011 ]
 [0.8356449  0.80062413 0.8548574  1.         0.8737456  0.8695341
  0.91274685 0.8508757 ]
 [0.9551031  0.89818645 0.9052491  0.8737456  1.0000002  0.87525815
  0.949548   0.9598906 ]
 [0.8825268  0.8449376  0.8240578  0.8695341  0.87525815 0.99999994
  0.94324416 0.8772442 ]
 [0.96892595 0.9529542  0.89955705 0.91274685 0.949548   0.94324416
  0.99999994 0.9796696 ]
 [0.9834153  0.9756224  0.8954011  0.8508757  0.9598906  0.8772442
  0.9796696  0.99999976]]
